# Cell 1: download your repo

In [ ]:
!wget -qO repo.zip https://github.com/AlinBolcas/Gemma4_FineTune_Creativity/archive/refs/heads/main.zip
!unzip -q repo.zip -d /kaggle/working
!rm -f repo.zip
!mv /kaggle/working/Gemma4_FineTune_Creativity-main /kaggle/working/Gemma4_FineTune_Creativity

Patch a fix:

In [ ]:
from pathlib import Path

repo = Path("/kaggle/working/Gemma4_FineTune_Creativity")
target = repo / "src" / "IV_inference" / "__init__.py"

target.write_text(
"""from .evaluate import evaluate, export_eval, load_eval_prompts
from .gemma4_integration import load_gemma4, load_finetuned_gemma4

try:
    from .ollama_integration import load_ollama_gemma4
except Exception:
    load_ollama_gemma4 = None
""",
encoding="utf-8",
)

print("Patched:", target)

# Cell 2: move into repo

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path("/kaggle/working/Gemma4_FineTune_Creativity")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

print(ROOT)

# Cell 3: install training deps

In [ ]:
!pip install -q -U unsloth datasets trl peft accelerate bitsandbytes huggingface_hub sentencepiece
!pip install -q "protobuf>=5.26,<6"

In [ ]:
modules = ["unsloth", "torch", "transformers", "datasets", "trl", "peft"]

for name in modules:
    try:
        __import__(name)
        print(f"OK: {name}")
    except Exception as e:
        print(f"FAIL: {name} -> {e}")

# Cell 4: load Hugging Face token

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import os

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)
os.environ["HUGGINGFACE_ACCESS_TOKEN"] = hf_token

print("HF token loaded")

# Cell 5: verify GPU

In [ ]:
import torch

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Cell 6: build a fresh Kaggle config
This makes a new config using Kaggle paths, not your Mac paths.

In [ ]:
from src.III_fineTune.sft_train import (
    discover_dataset_bundles,
    build_run_config,
    save_run_config,
    run_preflight,
)

bundles = discover_dataset_bundles()
print(bundles)

bundle = bundles[0]

config = build_run_config(
    bundle=bundle,
    model_alias="e2b",
    backend="unsloth",
    run_name="kaggle_smoke_e2b",
)

config["runtime"]["target"] = "kaggle"
config["training"]["output_dir"] = "data/output/models/kaggle_smoke_e2b"
config["training"]["num_train_epochs"] = 1
config["training"]["max_steps"] = 30
config["training"]["per_device_train_batch_size"] = 1
config["training"]["gradient_accumulation_steps"] = 4
config["training"]["max_seq_length"] = 4096

config_path = save_run_config(config)
print(config_path)

summary = run_preflight(config)
summary

# First training run
Use a tiny smoke train first.

Why:

proves the whole path works
produces a real adapter
gives you logs / metrics
cheap enough to fail safely

# Cell 7: train

In [ ]:
from src.III_fineTune.sft_train import train_from_config
train_from_config(config)

## 8. Post-Training Evaluation

After training finishes, use the cells below to:
- find the newest trained adapter
- load vanilla and fine-tuned Gemma
- compare outputs side by side
- run the held-out evaluation batch
- plot loss / learning-rate / gradient-norm curves

If you rerun this notebook from a freshly updated repo snapshot, the earlier patch cell for `src/IV_inference/__init__.py` should no longer be necessary.

In [ ]:
# Cell 8: find the latest adapter + training run
from pathlib import Path

models_dir = ROOT / "data" / "output" / "models"
runs_dir = ROOT / "data" / "output" / "training_runs"

adapter_dirs = sorted([p for p in models_dir.glob("*") if p.is_dir()], key=lambda p: p.stat().st_mtime, reverse=True)
run_dirs = sorted([p for p in runs_dir.glob("*") if p.is_dir()], key=lambda p: p.stat().st_mtime, reverse=True)

print("Adapters:")
for p in adapter_dirs[:10]:
    print(" -", p.name)

print("\nTraining runs:")
for p in run_dirs[:10]:
    print(" -", p.name)

adapter_path = adapter_dirs[0]
run_path = run_dirs[0]

print("\nUsing adapter:", adapter_path)
print("Using run dir:", run_path)

In [ ]:
# Cell 9: free memory before loading models for evaluation
import gc, torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("memory cleared")

In [ ]:
# Cell 11: quick direct comparison
prompt = "What project ideas would you generate for the Gemma 4 Good Hackathon that would genuinely impress judges?"

print("=== VANILLA ===")
print(vanilla("You are a helpful assistant.", prompt))

print("\n=== FINE-TUNED ===")
print(tuned("You are a helpful assistant.", prompt))

In [ ]:
# Cell 13: save a readable markdown evaluation report
import json

md_path = ROOT / "data/output/eval_kaggle_posttrain.md"

with open(eval_out, encoding="utf-8") as f:
    eval_rows = json.load(f)

lines = ["# Post-Training Evaluation", ""]
for i, row in enumerate(eval_rows, 1):
    lines.append(f"## Prompt {i}")
    lines.append("")
    lines.append(f"**Prompt:** {row['prompt']}")
    lines.append("")
    lines.append("### Tier 1 - Vanilla")
    lines.append("")
    lines.append(row["tier1_vanilla"])
    lines.append("")
    lines.append("### Tier 2 - Scaffolded Vanilla")
    lines.append("")
    lines.append(row["tier2_scaffolded"])
    lines.append("")
    lines.append("### Tier 3 - Fine-Tuned")
    lines.append("")
    lines.append(row["tier3_tuned"])
    lines.append("")
    lines.append("---")
    lines.append("")

md_path.write_text("\n".join(lines), encoding="utf-8")
print("Saved:", md_path)

In [ ]:
# Cell 15: plot loss / learning-rate / gradient norm
import matplotlib.pyplot as plt

if "loss" in df.columns and "step" in df.columns:
    plt.figure(figsize=(10, 5))
    train_df = df[df["loss"].notna()]
    plt.plot(train_df["step"], train_df["loss"], marker="o", label="train loss")
    if "eval_loss" in df.columns:
        eval_df = df[df["eval_loss"].notna()]
        if len(eval_df) > 0:
            plt.plot(eval_df["step"], eval_df["eval_loss"], marker="s", label="eval loss")
    plt.title("Training / Eval Loss")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.grid(True)
    plt.legend()
    plt.show()

if "learning_rate" in df.columns:
    lr_df = df[df["learning_rate"].notna()]
    if len(lr_df) > 0:
        plt.figure(figsize=(10, 4))
        plt.plot(lr_df["step"], lr_df["learning_rate"], marker=".")
        plt.title("Learning Rate")
        plt.xlabel("Step")
        plt.ylabel("LR")
        plt.grid(True)
        plt.show()

if "grad_norm" in df.columns:
    gn_df = df[df["grad_norm"].notna()]
    if len(gn_df) > 0:
        plt.figure(figsize=(10, 4))
        plt.plot(gn_df["step"], gn_df["grad_norm"], marker=".")
        plt.title("Gradient Norm")
        plt.xlabel("Step")
        plt.ylabel("Grad Norm")
        plt.grid(True)
        plt.show()